In [41]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [42]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
938,I haven't actually finished the film. You may ...,negative
761,I was very lucky to see this film as part of t...,positive
242,lovely. i just love the movie. i want to see t...,positive
366,I know that many people will/have automaticall...,negative
593,"Now I love Bela Lugosi,don't get me wrong,he i...",negative


In [43]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [44]:
df = normalize_text(df)
df.head()

,review,sentiment
938,actually finished film may say case right revi...,negative
761,lucky see film part melbourne international fi...,positive
242,lovely love movie want see movie rupert grint ...,positive
366,know many people will have automatically given...,negative
593,love bela lugosi don t get wrong he one intere...,negative


In [45]:
df['sentiment'].value_counts()

sentiment
positive    251
negative    249
Name: count, dtype: int64

In [46]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [47]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
938,actually finished film may say case right revi...,0
761,lucky see film part melbourne international fi...,1
242,lovely love movie want see movie rupert grint ...,1
366,know many people will have automatically given...,0
593,love bela lugosi don t get wrong he one intere...,0


In [48]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [49]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [51]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/zahir2003/Capstone-Project.mlflow')
dagshub.init(repo_owner='zahir2003', repo_name='Capstone-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2025-08-09 10:47:27,273 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/zahir2003/Capstone-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "zahir2003/Capstone-Project"

2025-08-09 10:47:27,279 - INFO - Initialized MLflow to track repo "zahir2003/Capstone-Project"


Repository zahir2003/Capstone-Project initialized!

2025-08-09 10:47:27,281 - INFO - Repository zahir2003/Capstone-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/75676ded29f84c298ca94fc426b7926c', creation_time=1754715669596, experiment_id='0', last_update_time=1754715669596, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [52]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2025-08-09 10:47:27,734 - INFO - Starting MLflow run...
2025-08-09 10:47:28,240 - INFO - Logging preprocessing parameters...
2025-08-09 10:47:29,679 - INFO - Initializing Logistic Regression model...
2025-08-09 10:47:29,680 - INFO - Fitting the model...
2025-08-09 10:47:29,690 - INFO - Model training complete.
2025-08-09 10:47:29,691 - INFO - Logging model parameters...
2025-08-09 10:47:30,160 - INFO - Making predictions...
2025-08-09 10:47:30,162 - INFO - Calculating evaluation metrics...
2025-08-09 10:47:30,171 - INFO - Logging evaluation metrics...
2025-08-09 10:47:31,787 - INFO - Saving and logging the model...
2025/08/09 10:47:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025-08-09 10:47:40,151 - INFO - Model training and logging completed in 11.91 seconds.
2025-08-09 10:47:40,153 - INFO - Accuracy: 0.68
2025-08-09 10:47:40,154 - INFO - Precision: 0

🏃 View run blushing-grub-105 at: https://dagshub.com/zahir2003/Capstone-Project.mlflow/#/experiments/0/runs/da94ab4657a4493bb7376377a9078d65
🧪 View experiment at: https://dagshub.com/zahir2003/Capstone-Project.mlflow/#/experiments/0
